# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** Air India — *Conditions of Carriage for Passengers, Baggage and Cargo* (legal contract of carriage)
**LLM:** Google Gemini (Flash) via the `google-genai` SDK
**Bi-Encoder:** `sentence-transformers/all-MiniLM-L6-v2` + FAISS
**Cross-Encoder:** `cross-encoder/ms-marco-MiniLM-L-6-v2`

Two pipelines run over the **exact same chunked corpus**:

| | Pipeline 1 — Baseline | Pipeline 2 — Re-Ranking |
|---|---|---|
| Stage 1 | Bi-Encoder → FAISS → Top 10 | *same Top 10* |
| Stage 2 | take Top 3 as-is | Cross-Encoder scores all 10 (query, chunk) pairs → re-sort → Top 3 |
| Generation | identical prompt → **Answer 1** | identical prompt → **Answer 2** |

**How to run:** put the PDF at `data/air-india-coc.pdf` (see README) and your key in a `.env` file as `GEMINI_API_KEY=...`, then *Run All*.

## 1. Setup & configuration

In [1]:
import os, re, json, time, hashlib, statistics
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                      # PDF text extraction
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss                        # vector store (exact inner-product search)
# NOTE: faiss must be imported AFTER torch. On macOS both ship their own OpenMP runtime,
# and importing faiss first makes the process crash silently on the first encode() call.
from dotenv import load_dotenv
from google import genai
from google.genai import types, errors

load_dotenv()                       # reads GEMINI_API_KEY from .env (never hard-code keys)
pd.set_option("display.max_colwidth", 120)

# ---- Experiment configuration (kept in one place so both pipelines share it) ----
PDF_PATH       = Path("data/air-india-coc.pdf")
CHUNK_WORDS    = 150    # target words per chunk
OVERLAP_WORDS  = 30     # words shared between consecutive chunks so sentences aren't cut off blindly
TOP_K          = 10     # candidates retrieved by the Bi-Encoder
TOP_N          = 3      # chunks passed to the LLM
BI_ENCODER     = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CACHE_PATH     = Path("cache/llm_answers.json")   # cached Gemini answers (saves free-tier quota)

# Run both models on CPU so latency numbers are comparable and reproducible on any laptop.
DEVICE = "cpu"
torch.manual_seed(0)
print("torch", torch.__version__, "| device:", DEVICE)

def realign_weights(model):
    # Copy every weight into freshly allocated memory.
    # Weights memory-mapped from .safetensors files are not 64-byte aligned, and on Apple Silicon
    # this made the Cross-Encoder's matrix multiplications return NaN for every score.
    with torch.no_grad():
        for p in model.parameters():
            p.data = p.data.clone()
    return model

torch 2.8.0 | device: cpu


## 2. Load the PDF

Text is extracted page by page. Every word keeps its page number so each chunk can be traced back to where it came from in the original document.

Two kinds of noise are removed first:
- the running header on every page (*"Air India Limited – Conditions of Carriage"*, *"Page N of 27"*), which would otherwise be repeated in almost every chunk;
- the table of contents (lines with `.....` dot leaders). It lists every article title, so it would act as an artificial distractor for almost every query.

In [2]:
NOISE_LINE = re.compile(r"^(Air India Limited – Conditions of Carriage|Page \d+ of \d+)$|\.{10,}")

def load_pdf_words(path):
    # Return a list of (word, page_number) for the whole document, with header/TOC lines removed.
    doc = pymupdf.open(path)
    words = []
    for page_no, page in enumerate(doc, start=1):
        for line in page.get_text("text").splitlines():
            line = line.strip()
            if not line or NOISE_LINE.search(line):
                continue
            words.extend((w, page_no) for w in line.split())
    return words, len(doc)

words, n_pages = load_pdf_words(PDF_PATH)
print(f"{n_pages} pages, {len(words):,} words")
print(" ".join(w for w, _ in words[:120]), "...")

27 pages, 12,542 words
AIR INDIA LIMITED CONDITIONS OF CARRIAGE FOR PASSENGERS, BAGGAGE AND CARGO Table of Contents Article 19 – CLAIMS UNDER EU REGULATION 261/2004 AND UK REGULATION 261/2004 27 Article 1 – DEFINITIONS “AGREED STOPPING PLACES” means those places, except the place of departure and the place of destination, set forth in the Ticket or shown in Carrier’s timetables as scheduled stopping places on the Passenger’s route. “BAGGAGE” means such articles, effects and other personal property of the Passenger as are necessary or appropriate for wear, use, comfort or convenience in connection with the trip and such other items as can be conveniently carried with the Passenger on such trip and are accepted by the Carrier for carriage. Unless otherwise specified, it ...


## 3. Chunking

A sliding window of `CHUNK_WORDS` words with `OVERLAP_WORDS` overlap. Chunks are deliberately small: in a legal contract a general rule and its *"except when…"* clause often end up in **different** chunks, which is exactly the situation where retrieval quality matters.

Both pipelines use this one list of chunks; nothing about the corpus differs between them.

In [3]:
def chunk_words(words, size, overlap):
    chunks, step = [], size - overlap
    for start in range(0, len(words), step):
        window = words[start:start + size]
        if len(window) < overlap and chunks:          # tiny tail already covered by the overlap
            break
        pages = sorted({p for _, p in window})
        chunks.append({
            "chunk_id": len(chunks),
            "pages": f"{pages[0]}" if len(pages) == 1 else f"{pages[0]}-{pages[-1]}",
            "text": " ".join(w for w, _ in window),
        })
    return chunks

chunks = chunk_words(words, CHUNK_WORDS, OVERLAP_WORDS)
chunk_texts = [c["text"] for c in chunks]
print(f"{len(chunks)} chunks")
pd.DataFrame(chunks).head()

105 chunks


,chunk_id,pages,text
0,0,1-2,"AIR INDIA LIMITED CONDITIONS OF CARRIAGE FOR PASSENGERS, BAGGAGE AND CARGO Table of Contents Article 19 – CLAIMS UND..."
1,1,2,includes both Checked and Unchecked Baggage of the Passenger. “BAGGAGE CHECK” means those portions of the Ticket whi...
2,2,2,which the Carrier takes sole custody and for which the Carrier has issued a Baggage Check. “COMMERCIAL AGREEMENT” me...
3,3,2-3,"upon which notice is dispatched shall not be counted; and that, for purposes of determining duration of validity, th..."
4,4,3,Electronic Ticket which contains the information and notices required under the Convention or as otherwise required....


## 4. Pipeline 1, stage 1: Bi-Encoder index (FAISS)

The Bi-Encoder embeds every chunk **once, offline**. Vectors are L2-normalised, so the inner product computed by `IndexFlatIP` is exactly **cosine similarity**. At query time only the query is embedded and compared against the stored vectors.

In [4]:
bi_encoder = realign_weights(SentenceTransformer(BI_ENCODER, device=DEVICE))

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=32, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
index_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_emb.shape[1])   # exact search; cosine because vectors are normalised
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]}, offline indexing took {index_time:.2f}s")

Indexed 105 chunks, dim=384, offline indexing took 1.08s


In [5]:
def retrieve(query, k=TOP_K):
    # Bi-Encoder retrieval: returns the top-k chunks by cosine similarity, plus stage timings.
    t0 = time.perf_counter()
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    t1 = time.perf_counter()
    scores, ids = index.search(q, k)
    t2 = time.perf_counter()
    hits = [{"chunk_id": int(i), "bi_rank": r + 1, "bi_score": float(s)}
            for r, (i, s) in enumerate(zip(ids[0], scores[0]))]
    return hits, {"embed_query": t1 - t0, "faiss_search": t2 - t1}

## 5. Pipeline 2, stage 2: Cross-Encoder re-ranking

The Cross-Encoder reads the query and a chunk **together** as a single input (`[CLS] query [SEP] chunk [SEP]`), so every query token can attend to every chunk token. It outputs one relevance score per pair. That makes it more accurate, but nothing can be pre-computed, so it is applied only to the 10 candidates from stage 1.

In [6]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE)
realign_weights(cross_encoder.model)

# Sanity check: scores must be real numbers, and a relevant passage must outscore an irrelevant one.
_check = cross_encoder.predict([("Can I bring my dog?", "Pets such as dogs and cats are accepted in a crate."),
                                ("Can I bring my dog?", "Refunds are paid in the original currency.")])
assert not np.isnan(_check).any() and _check[0] > _check[1], _check

def rerank(query, hits):
    # Score all (query, chunk) pairs with the Cross-Encoder and sort descending by that score.
    t0 = time.perf_counter()
    pairs = [(query, chunk_texts[h["chunk_id"]]) for h in hits]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    elapsed = time.perf_counter() - t0
    reranked = [dict(h, ce_score=float(s)) for h, s in zip(hits, ce_scores)]
    reranked.sort(key=lambda h: h["ce_score"], reverse=True)
    for r, h in enumerate(reranked, start=1):
        h["ce_rank"] = r
    return reranked, {"cross_encoder": elapsed}

## 6. Generation with Gemini

Both pipelines use the **identical** prompt template and settings (`temperature=0`), so any difference between Answer 1 and Answer 2 comes only from which chunks were retrieved.

Answers are cached on disk, keyed by model + prompt, so re-running the notebook doesn't use up free-tier quota.

In [7]:
# The model is pinned rather than auto-picked: cached answers are keyed by model name, so a newer
# Flash release would otherwise silently miss the cache. Override with GEMINI_MODEL in .env.
LLM_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
print("Using LLM:", LLM_MODEL)

# No API key is needed when every answer is already cached (re-running the recorded experiment).
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"]) if os.getenv("GEMINI_API_KEY") else None

PROMPT_TEMPLATE = '''You are answering questions about an airline's Conditions of Carriage.
Use ONLY the context excerpts below. If the context does not contain the answer, say so explicitly.
Quote or cite the relevant rule, including any conditions or exceptions.

Context:
{context}

Question: {question}
Answer:'''

def build_prompt(question, hits):
    context = "\n\n".join(f"[Excerpt {i} | chunk {h['chunk_id']} | p.{chunks[h['chunk_id']]['pages']}]\n"
                          f"{chunk_texts[h['chunk_id']]}" for i, h in enumerate(hits, start=1))
    return PROMPT_TEMPLATE.format(context=context, question=question)

_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

def generate(prompt, max_retries=5):
    # Call Gemini (temperature 0). Returns (answer, seconds, from_cache). Retries on rate limits.
    key = hashlib.sha256(f"{LLM_MODEL}\n{prompt}".encode()).hexdigest()
    if key in _cache:
        return _cache[key]["answer"], _cache[key]["seconds"], True
    if client is None:
        raise RuntimeError("Answer is not cached and GEMINI_API_KEY is not set (see .env.example)")
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            resp = client.models.generate_content(
                model=LLM_MODEL, contents=prompt,
                config=types.GenerateContentConfig(temperature=0.0))
            seconds = time.perf_counter() - t0
            break
        except errors.APIError as e:
            if e.code in (429, 503) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Gemini {e.code}, retrying in {wait}s ...")
                time.sleep(wait)
            else:
                raise
    _cache[key] = {"answer": resp.text, "seconds": seconds}
    CACHE_PATH.parent.mkdir(exist_ok=True)
    CACHE_PATH.write_text(json.dumps(_cache, indent=1))
    return resp.text, seconds, False

Using LLM: gemini-3.8-flash


## 7. The two pipelines end to end

`pipeline_1` and `pipeline_2` share stage 1. The only difference is the Cross-Encoder step before choosing the Top 3.

In [8]:
def pipeline_1(query, with_llm=True):
    hits, t = retrieve(query)
    top = hits[:TOP_N]
    out = {"hits": hits, "top": top, "timings": t}
    if with_llm:
        out["answer"], t["llm"], out["cached"] = generate(build_prompt(query, top))
    return out

def pipeline_2(query, with_llm=True):
    hits, t = retrieve(query)
    reranked, t_ce = rerank(query, hits)
    t.update(t_ce)
    top = reranked[:TOP_N]
    out = {"hits": reranked, "top": top, "timings": t}
    if with_llm:
        out["answer"], t["llm"], out["cached"] = generate(build_prompt(query, top))
    return out

def rank_table(query, gold_ids=()):
    # Side-by-side view of the 10 candidates: Bi-Encoder rank vs Cross-Encoder rank.
    reranked, _ = rerank(query, retrieve(query)[0])
    rows = []
    for h in sorted(reranked, key=lambda h: h["bi_rank"]):
        rows.append({
            "chunk": h["chunk_id"], "page": chunks[h["chunk_id"]]["pages"],
            "bi_rank": h["bi_rank"], "bi_score": round(h["bi_score"], 3),
            "ce_rank": h["ce_rank"], "ce_score": round(h["ce_score"], 2),
            "shift": f"#{h['bi_rank']} → #{h['ce_rank']}",
            "gold": "★" if h["chunk_id"] in gold_ids else "",
            "snippet": chunk_texts[h["chunk_id"]][:110] + "…",
        })
    return pd.DataFrame(rows).set_index("chunk")

## 8. Failure-mode stress tests

Each test query has a **gold chunk**: the chunk that actually contains the answer. It is identified by a distinctive phrase from the PDF instead of a hard-coded chunk ID, so it stays correct if the chunk size is changed.

Categories (from the assignment):
- **Keyword trap / distractor**: the query shares words with an irrelevant passage, while the real answer uses different terms.
- **Exception / corner case**: the general rule is easy to find, but the *"except when…"* clause is elsewhere.
- **Negation / constraint**: "not allowed", "prohibited", "refuse": Bi-Encoder embeddings blur negative and positive context.
- **Buried chunk**: the vital chunk sits at rank 4–10 and needs to be promoted.

This first pass uses **retrieval only, with no LLM calls**, so many candidate queries can be screened for free.

In [9]:
# Each test query: id, failure category, the question, and a distinctive phrase from the gold chunk.
TEST_QUERIES = [
    # --- Keyword trap / distractor: query wording overlaps with the wrong section ---
    {"id": "KT1", "category": "Keyword trap", "query": "Will I get a refund if my seat is downgraded to a lower class?", "gold_phrase": "involuntarily downgraded, in domestic carriage"},
    {"id": "KT2", "category": "Keyword trap", "query": "How much compensation do I get if my bag is lost on a domestic flight?", "gold_phrase": "lost Checked Baggage will be Rupees Twenty-Five Thousand"},
    {"id": "KT3", "category": "Keyword trap", "query": "What is the maximum compensation if a passenger dies on a domestic flight?", "gold_phrase": "Twenty-Five Lakhs and Fifty Thousand Rupees"},
    {"id": "KT4", "category": "Keyword trap", "query": "Can I cancel my ticket for free right after booking it?", "gold_phrase": "look-in option"},
    {"id": "KT5", "category": "Keyword trap", "query": "Is there an extra fee for booking on the Air India website?", "gold_phrase": "convenience fee is a non-refundable"},
    {"id": "KT6", "category": "Keyword trap", "query": "What happens if the airline overbooks my flight?", "gold_phrase": "more Passengers reporting for a particular flight than available seats"},
    {"id": "KT7", "category": "Keyword trap", "query": "I want to get off the plane after boarding, will I be charged?", "gold_phrase": "levy a de-boarding charge"},
    {"id": "KT8", "category": "Keyword trap", "query": "Can I take my dog on the flight with me?", "gold_phrase": "household birds and other pets"},
    # --- Exceptions & corner cases: general rule found, "except when..." missed ---
    {"id": "EX1", "category": "Exception", "query": "Will Air India refund my ticket if my international flight is delayed by more than six hours?", "gold_phrase": "shall not be liable to refund the value of the Ticket"},
    {"id": "EX2", "category": "Exception", "query": "Does the airline pay compensation when a flight is cancelled because of bad weather?", "gold_phrase": "meteorological conditions"},
    {"id": "EX3", "category": "Exception", "query": "Can I still get cancellation compensation if I gave the airline a wrong phone number when booking?", "gold_phrase": "inadequate contact information"},
    {"id": "EX4", "category": "Exception", "query": "Can I use the 48-hour free cancellation for a domestic flight that departs in five days?", "gold_phrase": "less than seven (7) days from the booking"},
    {"id": "EX5", "category": "Exception", "query": "Will Air India pay for my suitcase if its wheels broke during the flight?", "gold_phrase": "broken wheels or base"},
    {"id": "EX6", "category": "Exception", "query": "Is the airline liable if jewellery packed in my checked bag goes missing?", "gold_phrase": "items that are not permitted to be carried as Checked Baggage"},
    {"id": "EX7", "category": "Exception", "query": "Do I have to pay baggage charges for my guide dog?", "gold_phrase": "Guide dogs accompanying"},
    {"id": "EX8", "category": "Exception", "query": "Can the de-boarding charge be waived if a family member dies?", "gold_phrase": "namely spouse, child, parent"},
    {"id": "EX9", "category": "Exception", "query": "Can my ticket validity be extended if a family member dies during my trip?", "gold_phrase": "forty-five (45) days from the date of the death"},
    {"id": "EX10", "category": "Exception", "query": "Does a pet travelling with me count towards my free baggage allowance?", "gold_phrase": "shall not be included in the free Baggage allowance"},
    {"id": "EX11", "category": "Exception", "query": "How many days do I have to complain about damaged checked baggage?", "gold_phrase": "within seven (7) days from the date of receipt"},
    {"id": "EX12", "category": "Exception", "query": "How many days do I have to complain about delayed baggage?", "gold_phrase": "twenty-one (21) days"},
    # --- Negation & constraints: "not allowed", "prohibited", "no authority" ---
    {"id": "NG1", "category": "Negation", "query": "What items am I not allowed to pack in my checked baggage?", "gold_phrase": "shall not include in Checked Baggage fragile"},
    {"id": "NG2", "category": "Negation", "query": "Are firearms prohibited on Air India flights?", "gold_phrase": "Firearms and ammunition, other than for hunting"},
    {"id": "NG3", "category": "Negation", "query": "Who is not allowed to change or waive the conditions of carriage?", "gold_phrase": "No agent, employee or representative of the Carrier has authority"},
    {"id": "NG4", "category": "Negation", "query": "Is a passenger who misbehaves on board not entitled to a refund?", "gold_phrase": "shall not be entitled to any refunds"},
    {"id": "NG5", "category": "Negation", "query": "Does an unruly passenger have to pay if the plane is diverted because of them?", "gold_phrase": "divert the aircraft for the purpose of offloading"},
    {"id": "NG6", "category": "Negation", "query": "In which cases can Air India refuse to give me a refund?", "gold_phrase": "may refuse refund on a Ticket which has been presented"},
    {"id": "NG7", "category": "Negation", "query": "When will the airline not pay compensation for denied boarding?", "gold_phrase": "alternate flight scheduled to depart within one hour"},
    {"id": "NG8", "category": "Negation", "query": "Can I bring a delicate musical instrument into the cabin?", "gold_phrase": "delicate musical instruments"},
    {"id": "NG9", "category": "Negation", "query": "Is my fare refunded if I am refused entry into a country and sent back?", "gold_phrase": "point of refusal of entry or deportation will not be refunded"},
    {"id": "NG10", "category": "Negation", "query": "Will the airline refuse to hand over my bag if I lost my baggage tag?", "gold_phrase": "shall not prevent delivery"},
]

In [10]:
def gold_chunk_ids(phrase):
    # All chunks containing the phrase (case/whitespace-insensitive). Overlap can put it in 2 chunks.
    norm = lambda s: re.sub(r"\s+", " ", s).lower()
    return {c["chunk_id"] for c in chunks if norm(phrase) in norm(c["text"])}

screen = []
for tq in TEST_QUERIES:
    gold = gold_chunk_ids(tq["gold_phrase"])
    assert gold, f"gold phrase not found in any chunk: {tq['gold_phrase']!r}"
    reranked, _ = rerank(tq["query"], retrieve(tq["query"])[0])
    bi = min((h["bi_rank"] for h in reranked if h["chunk_id"] in gold), default=None)
    ce = min((h["ce_rank"] for h in reranked if h["chunk_id"] in gold), default=None)
    screen.append({"id": tq["id"], "category": tq["category"], "query": tq["query"],
                   "gold_chunks": sorted(gold), "bi_rank": bi if bi else ">10", "ce_rank": ce if ce else "-",
                   "baseline_top3": bool(bi and bi <= TOP_N), "rerank_top3": bool(ce and ce <= TOP_N)})

screen_df = pd.DataFrame(screen).set_index("id")
screen_df["rerank_fixes_it"] = ~screen_df.baseline_top3 & screen_df.rerank_top3
screen_df

,category,query,gold_chunks,bi_rank,ce_rank,baseline_top3,rerank_top3,rerank_fixes_it
id,,,,,,,,
KT1,Keyword trap,Will I get a refund if my seat is downgraded to a lower class?,[24],1,1,True,True,False
KT2,Keyword trap,How much compensation do I get if my bag is lost on a domestic flight?,[86],>10,-,False,False,False
KT3,Keyword trap,What is the maximum compensation if a passenger dies on a domestic flight?,[78],4,3,False,True,True
KT4,Keyword trap,Can I cancel my ticket for free right after booking it?,[21],5,1,False,True,True
KT5,Keyword trap,Is there an extra fee for booking on the Air India website?,[18],3,6,True,False,False
KT6,Keyword trap,What happens if the airline overbooks my flight?,"[34, 35]",>10,-,False,False,False
KT7,Keyword trap,"I want to get off the plane after boarding, will I be charged?",[37],1,1,True,True,False
KT8,Keyword trap,Can I take my dog on the flight with me?,[47],2,3,True,True,False
EX1,Exception,Will Air India refund my ticket if my international flight is delayed by more than six hours?,[57],1,1,True,True,False


In [11]:
print("Baseline gets gold into Top 3:", screen_df.baseline_top3.sum(), "/", len(screen_df))
print("Re-ranking gets gold into Top 3:", screen_df.rerank_top3.sum(), "/", len(screen_df))
print("Fixed by re-ranking (baseline miss -> rerank hit):", list(screen_df.index[screen_df.rerank_fixes_it]))
print("Hurt by re-ranking (baseline hit -> rerank miss):",
      list(screen_df.index[screen_df.baseline_top3 & ~screen_df.rerank_top3]))

Baseline gets gold into Top 3: 21 / 30
Re-ranking gets gold into Top 3: 26 / 30
Fixed by re-ranking (baseline miss -> rerank hit): ['KT3', 'KT4', 'EX3', 'NG3', 'NG5', 'NG7']
Hurt by re-ranking (baseline hit -> rerank miss): ['KT5']


## 9. Showcase: rank shifts and Answer 1 vs Answer 2

For each showcase query: the full rank table for the 10 candidates (★ = gold chunk), then both LLM answers. Answers are only generated for these few queries to stay within the Gemini free tier.

In [12]:
from IPython.display import display, Markdown

def showcase(query_id):
    tq = next(t for t in TEST_QUERIES if t["id"] == query_id)
    gold = gold_chunk_ids(tq["gold_phrase"])
    display(Markdown(f"### {tq['id']} — {tq['category']}\n**Query:** {tq['query']}\n\n"
                     f"**Gold chunk(s):** {sorted(gold)}"))
    display(rank_table(tq["query"], gold))
    p1, p2 = pipeline_1(tq["query"]), pipeline_2(tq["query"])
    display(Markdown(
        f"**Pipeline 1 Top 3:** {[h['chunk_id'] for h in p1['top']]} &nbsp;|&nbsp; "
        f"**Pipeline 2 Top 3:** {[h['chunk_id'] for h in p2['top']]}\n\n"
        f"#### Answer 1 (baseline)\n{p1['answer']}\n\n#### Answer 2 (re-ranked)\n{p2['answer']}\n\n---"))
    if not (p1["cached"] and p2["cached"]):
        time.sleep(6)          # stay under the free-tier requests-per-minute limit

# All queries re-ranking fixed, plus any it made worse (counter-examples are shown too).
SHOWCASE_IDS = (list(screen_df.index[screen_df.rerank_fixes_it])
                + list(screen_df.index[screen_df.baseline_top3 & ~screen_df.rerank_top3]))
for qid in SHOWCASE_IDS:
    showcase(qid)

### KT3 — Keyword trap
**Query:** What is the maximum compensation if a passenger dies on a domestic flight?

**Gold chunk(s):** [78]

,page,bi_rank,bi_score,ce_rank,ce_score,shift,gold,snippet
chunk,,,,,,,,
76,21,1,0.555,2,-0.54,#1 → #2,,"be liable for damages sustained in case of death or bodily injury of a Passenger, upon condition only that the…"
77,21-22,2,0.542,1,3.95,#2 → #1,,"carrier box of the Ticket for that flight or flight segment, and to the limits on compensation as provided und…"
98,26,3,0.526,6,-3.93,#3 → #6,,for damages in case of death may be brought in the above-mentioned courts or in the territory of a State Party…
78,22,4,0.496,3,-1.24,#4 → #3,★,"a maximum of 125,000 Francs which, being a now invalid currency, shall be deemed to be the equivalent of 128,8…"
14,5-6,5,0.476,5,-3.72,#5 → #5,,"from the date shown on such certificate. In such circumstances, the Carrier may extend similarly the period of…"
95,25-26,6,0.458,10,-10.29,#6 → #10,,"Carrier shall apply to and be for the benefit of agents, employees and representatives of the Carrier and any …"
82,22-23,7,0.454,8,-6.74,#7 → #8,,"be deemed to have the monetary equivalent of USD 20 per kilogram, unless the national laws of the country of t…"
60,17,8,0.448,4,-2.43,#8 → #4,,"or a connecting flight is caused to be missed owing to cancellation, then in addition to meals and refreshment…"
97,26,9,0.433,9,-9.27,#9 → #9,,"at the destination, or from the date on which the aircraft ought to have arrived, or from the date on which th…"


**Pipeline 1 Top 3:** [76, 77, 98] &nbsp;|&nbsp; **Pipeline 2 Top 3:** [77, 76, 78]

#### Answer 1 (baseline)
Based on the provided context, there is no mention of the maximum compensation for a passenger's death on a **domestic flight**. 

The excerpts only provide compensation limits for **international carriage**:
* **Article 17.1.1:** *"For international carriage to which the Montreal Convention applies, the Carrier’s liability in the case of death or bodily injury to Passengers will be limited up to a maximum of 128,821 SDRs, unless a higher monetary limit has been incorporated into the national laws of the country of the court seized of the case."*
* **Article 17.1.2:** *"For international carriage to which the Warsaw Convention applies, the Carrier’s liability in the case of death or bodily injury to Passengers will be limited up to a maximum of 125,000 Francs which, being a now invalid currency, shall be deemed to be the equivalent of 128,821 SDRs, unless the national laws of the country of the [court seized of the case provide otherwise]."*

#### Answer 2 (re-ranked)
Based on the provided excerpts, for domestic flights (referred to as "Non-International Carriage"), the compensation limits and conditions are as follows:

* **Maximum Limit:** Under **Article 17.1.3**, *"the Carrier’s liability in the case of death or bodily injury to Passengers will be limited to the sum of Twenty-Five Lakhs and Fifty Thousand Rupees, unless a higher monetary limit has been incorporated into law by the Government of India."*
* **Exceptions regarding limits:** Under **Article 17.1.4**, *"To the extent that damages under Articles 17.1.1, 17.1.2 and 17.1.3 may potentially exceed their respective stated limits, the Carrier shall not be liable for any sums in excess of those figures if the Carrier proves that the Damage (being the death or bodily injury) was not due to the negligence or other wrongful act or omission of the Carrier or its servants or..."*
* **General Conditions:** 
  * Under **Article 16.1**, liability applies *"upon condition only that the accident which caused the death or injury took place on board the aircraft, or in the course of any of the operations of embarking or disembarking."*
  * Under **Article 16.3**, compensation is subject to *"the Airline Designator Code of Air India appearing in the carrier box of the Ticket for that flight or flight segment, and to the limits on compensation as provided under these Conditions of Carriage."*

---

### KT4 — Keyword trap
**Query:** Can I cancel my ticket for free right after booking it?

**Gold chunk(s):** [21]

,page,bi_rank,bi_score,ce_rank,ce_score,shift,gold,snippet
chunk,,,,,,,,
22,7-8,1,0.526,3,0.36,#1 → #3,,"booking and the revised booking. For all domestic bookings, and bookings for travel from the USA or Canada, th…"
26,8,2,0.518,4,-0.62,#2 → #4,,reservation in accordance with and within the time limits specified in the Carrier’s Regulations. Failure to c…
25,8,3,0.508,7,-6.67,#3 → #7,,"a class below that for which the Ticket was purchased, the Carrier shall reimburse the Passenger: (i) 30% of t…"
55,16,4,0.507,6,-6.11,#4 → #6,,surrender to the Carrier of the Passenger Coupon or Passenger Receipt and surrender of all unused Flight Coupo…
21,7,5,0.503,1,4.87,#5 → #1,★,"its authorised agent. 6.1.2 As provided in the Carrier’s Regulations, certain fares may have conditions which …"
54,16,6,0.500,10,-8.25,#6 → #10,,"the Carrier shall be entitled to make refund either to the person named in the Ticket, or to the person who ha…"
63,18,7,0.470,5,-4.79,#7 → #5,,which could not be avoided even though the Carrier had taken all reasonable measures to avoid or overcome the …
56,16-17,8,0.467,9,-7.88,#8 → #9,,"connecting flight on which the Passenger holds a reservation, the amount of the refund shall be: 11.3.1 if no …"
64,18,9,0.465,8,-7.44,#9 → #8,,"thereof is lost, refund will be made on proof of loss satisfactory to the Carrier and upon payment of any appl…"


...

**Pipeline 1 Top 3:** [29, 100, 61] &nbsp;|&nbsp; **Pipeline 2 Top 3:** [36, 35, 29]

#### Answer 1 (baseline)
Based on the provided context, the excerpts do not contain specific information regarding when the airline will not pay compensation for **denied boarding**. 

The context only mentions:
* The right to deny boarding if a payment transaction is unverified for a travel agency booking and the passenger fails to purchase a fresh ticket directly from the Carrier (Excerpt 1, Rule 6.8.4).
* General circumstances where compensation is not paid for *flight cancellations or delays* (such as force majeure events) or where "inadequate contact information has been provided at the time of booking" (Excerpt 3).

#### Answer 2 (re-ranked)
Based on the provided excerpts, the airline is not liable to pay compensation for denied boarding under the following condition:

* When passengers are denied boarding because more passengers report for a flight than available seats, and **"the Carrier arranges an alternate flight scheduled to depart within one hour from the originally scheduled departure time."** ([Excerpt 2 | chunk 35 | p.11])

---

### KT5 — Keyword trap
**Query:** Is there an extra fee for booking on the Air India website?

**Gold chunk(s):** [18]

,page,bi_rank,bi_score,ce_rank,ce_score,shift,gold,snippet
chunk,,,,,,,,
21,7,1,0.593,1,1.35,#1 → #1,,"its authorised agent. 6.1.2 As provided in the Carrier’s Regulations, certain fares may have conditions which …"
19,6-7,2,0.539,3,-2.79,#2 → #3,,"a non-refundable fee of INR 300 that is applied to all domestic bookings made via the Carrier’s website, mobil…"
18,6-7,3,0.528,6,-5.03,#3 → #6,★,"Applicable fares are those published by or on behalf of the Carrier or, if not so published, constructed in ac…"
20,7,4,0.518,10,-10.53,#4 → #10,,"by a Passenger of any service or facilities, will be in addition to the published fares and charges, and shall…"
16,6,5,0.486,2,-2.14,#5 → #2,,"shall apply to all Tickets issued on an Air India (098) document, for domestic as well as international travel…"
60,17,6,0.483,4,-4.31,#6 → #4,,"or a connecting flight is caused to be missed owing to cancellation, then in addition to meals and refreshment…"
22,7-8,7,0.455,5,-4.57,#7 → #5,,"booking and the revised booking. For all domestic bookings, and bookings for travel from the USA or Canada, th…"
25,8,8,0.437,9,-9.31,#8 → #9,,"a class below that for which the Ticket was purchased, the Carrier shall reimburse the Passenger: (i) 30% of t…"
15,5-6,9,0.402,8,-6.79,#9 → #8,,of validity shall not be for a period longer than forty-five (45) days from the date of the death. 3.3 FLIGHT …


  Gemini 503, retrying in 20s ...


**Pipeline 1 Top 3:** [21, 19, 18] &nbsp;|&nbsp; **Pipeline 2 Top 3:** [21, 16, 19]

#### Answer 1 (baseline)
Based on the provided excerpts, yes, there is an extra fee for domestic bookings made on the Carrier’s website:

* **Rule / Fee:** "The convenience fee is a non-refundable fee of INR 300 that is applied to all domestic bookings made via the Carrier’s website, mobile application and call centres." [Excerpt 2 & 3 | p.6-7]
* **Conditions:** 
  * "The fee applies on all modes of payment and is charged per customer per Ticket." [Excerpt 2 | chunk 19 | p.6-7]
  * It is explicitly stated for **domestic bookings**; the provided context does not mention whether a fee applies to international bookings made on the website.

#### Answer 2 (re-ranked)
Yes, for domestic bookings, there is an extra fee. 

According to Excerpt 3 (p. 6–7), there is:
> "...a non-refundable fee of INR 300 that is applied to all domestic bookings made via the Carrier’s website, mobile application and call centres."

**Conditions:**
* "The fee applies on all modes of payment and is charged per customer per Ticket."

*(Note: The provided text does not mention whether an extra fee applies to international bookings made on the website.)*

---

## 10. Latency profiling

Each stage is timed with `time.perf_counter()` after a warm-up call, averaged over `N_RUNS` runs for every test query. LLM time is reported separately because it is dominated by network latency and is the same kind of call in both pipelines.

In [13]:
N_RUNS = 20
for _ in range(3):                                   # warm-up: first calls include one-off setup cost
    pipeline_2(TEST_QUERIES[0]["query"], with_llm=False)

rows = []
for tq in TEST_QUERIES:
    for _ in range(N_RUNS):
        t1 = pipeline_1(tq["query"], with_llm=False)["timings"]
        t2 = pipeline_2(tq["query"], with_llm=False)["timings"]
        rows.append({"pipeline": "1: Baseline (bi-encoder + FAISS)",
                     **{k: v * 1000 for k, v in t1.items()}})
        rows.append({"pipeline": "2: Baseline + Cross-Encoder re-rank",
                     **{k: v * 1000 for k, v in t2.items()}})

lat = pd.DataFrame(rows).fillna(0.0)
lat["retrieval_total"] = lat[["embed_query", "faiss_search", "cross_encoder"]].sum(axis=1)
def p95(s):
    return s.quantile(0.95)

latency_table = lat.groupby("pipeline").agg(["mean", "median", p95]).round(2)
print(f"Retrieval latency in ms ({N_RUNS} runs x {len(TEST_QUERIES)} queries, CPU)")
latency_table

Retrieval latency in ms (20 runs x 30 queries, CPU)


embed_query              faiss_search  \
                                           mean median   p95         mean   
pipeline                                                                    
1: Baseline (bi-encoder + FAISS)           5.35   5.03  6.36         0.03   
2: Baseline + Cross-Encoder re-rank        4.60   4.38  5.66         0.02   

                                                 cross_encoder          \
                                    median   p95          mean  median   
pipeline                                                                 
1: Baseline (bi-encoder + FAISS)      0.02  0.03          0.00    0.00   
2: Baseline + Cross-Encoder re-rank   0.02  0.03        116.46  113.61   

                                            retrieval_total                  
                                        p95            mean  median     p95  
pipeline                                                                     
1: Baseline (bi-encoder + FAISS)       0.00            5.37    5.06    6.39  
2: Baseline + Cross-Encoder re-rank  148.09          121.09  118.05  153.45

In [14]:
# LLM generation time, measured on the real (uncached) Gemini calls made in section 9.
llm_secs = [v["seconds"] for v in _cache.values()]
if llm_secs:
    print(f"Gemini generation ({len(llm_secs)} calls): mean {statistics.mean(llm_secs):.2f}s, "
          f"median {statistics.median(llm_secs):.2f}s")

Gemini generation (14 calls): mean 6.25s, median 5.88s


## 11. Scaling: why not run the Cross-Encoder over everything?

To make the scaling argument concrete: score **every** chunk in this document with the Cross-Encoder for a single query, then extrapolate the per-pair cost to larger collections.

In [15]:
q = TEST_QUERIES[0]["query"]
all_pairs = [(q, t) for t in chunk_texts]

t0 = time.perf_counter(); cross_encoder.predict(all_pairs[:TOP_K], show_progress_bar=False)
t_top10 = time.perf_counter() - t0
t0 = time.perf_counter(); cross_encoder.predict(all_pairs, batch_size=32, show_progress_bar=False)
t_all = time.perf_counter() - t0
per_pair = t_all / len(all_pairs)

t0 = time.perf_counter()
for _ in range(100):
    retrieve(q)
t_bi = (time.perf_counter() - t0) / 100

scale = pd.DataFrame({
    "corpus_size (chunks)": [len(chunks), 10_000, 1_000_000, 100_000_000],
})
scale["cross_encoder_on_all (s)"] = (scale["corpus_size (chunks)"] * per_pair).round(2)
scale["bi_encoder_top10 + rerank_10 (s)"] = round(t_bi + t_top10, 3)   # FAISS search barely grows (ANN indexes are sub-linear)
print(f"Measured: Cross-Encoder on top-10 = {t_top10*1000:.1f} ms | on all {len(chunks)} chunks = {t_all*1000:.1f} ms "
      f"| {per_pair*1000:.2f} ms per pair")
scale

Measured: Cross-Encoder on top-10 = 107.1 ms | on all 105 chunks = 1251.5 ms | 11.92 ms per pair


,corpus_size (chunks),cross_encoder_on_all (s),bi_encoder_top10 + rerank_10 (s)
0,105,1.25,0.111
1,10000,119.19,0.111
2,1000000,11919.01,0.111
3,100000000,1191901.03,0.111


---
*The written analysis (architecture comparison, rank-shift discussion, latency and scaling) is in the accompanying report.*